In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

df = pd.read_csv('results.csv.gz')
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Saving results.csv.gz to results.csv.gz
Rows: 15739
Columns: ['id', 'name', 'isFork', 'commits', 'branches', 'releases', 'forks', 'mainLanguage', 'defaultBranch', 'license', 'homepage', 'watchers', 'stargazers', 'contributors', 'size', 'createdAt', 'pushedAt', 'updatedAt', 'totalIssues', 'openIssues', 'totalPullRequests', 'openPullRequests', 'blankLines', 'codeLines', 'commentLines', 'metrics', 'lastCommit', 'lastCommitSHA', 'hasWiki', 'isArchived', 'isDisabled', 'isLocked', 'languages', 'labels', 'topics']


,id,name,isFork,commits,branches,releases,forks,mainLanguage,defaultBranch,license,...,metrics,lastCommit,lastCommitSHA,hasWiki,isArchived,isDisabled,isLocked,languages,labels,topics
0,17158847,bkerler/mtkclient,False,104,2,3,133,Python,main,GNU General Public License v3.0,...,"[{""blankLines"":3,""codeLines"":13,""commentLines""...",2026-06-09T08:04:27,2c9f4d78601e2b223cacfed773a5c4cbb1808189,True,False,False,False,"{""Python"":1668194,""C"":63290,""Makefile"":1816,""C...",bug;documentation;duplicate;enhancement;good f...,NaN
1,39089105,pwncollege/challenges,False,330,21,0,28,Python,main,NaN,...,"[{""blankLines"":2669,""codeLines"":10737,""comment...",2026-06-19T01:16:00,b4a3ddbd5ed70467ae2cbe16dcd5f43472e8769a,True,False,False,False,"{""Python"":497756,""Jinja"":362391,""C"":104790,""Sh...",bug;dangerous;documentation;duplicate;enhancem...,NaN
2,48237758,cs-ubbcluj-ro/fp,False,45,1,0,3,Python,main,NaN,...,"[{""blankLines"":0,""codeLines"":3,""commentLines"":...",2026-01-17T08:34:35,d28c7593a946771370825e97985a9bc17aaeb1a8,True,False,False,False,"{""Python"":297915,""HTML"":56342}",bug;documentation;duplicate;enhancement;good f...,NaN
3,70370698,iic2233/syllabus,False,82,1,0,11,Python,main,NaN,...,"[{""blankLines"":0,""codeLines"":97,""commentLines""...",2026-06-18T06:49:47,a421764ec0d14a8ccf52d7febb450cdcbf654db9,True,False,False,False,"{""Python"":699448,""Jupyter Notebook"":625316,""Ja...",bug;duplicada;importante,NaN
4,94720502,allenai/OLMo-Eval,False,423,60,0,5,Python,main,Apache License 2.0,...,"[{""blankLines"":20,""codeLines"":72,""commentLines...",2026-06-16T04:30:26,b0cb464f19dbe0cbcfddd8d19afab9cc8257d99c,True,False,False,False,"{""Python"":3316101,""JavaScript"":127506,""Shell"":...",bug;dependencies;documentation;duplicate;enhan...,NaN


In [ ]:
keep = [
    "MIT License",
    "Apache License 2.0",
    'BSD 2-Clause "Simplified" License',
    'BSD 3-Clause "New" or "Revised" License',
]

filtered = df[df["license"].isin(keep)]

print("Before license filter:", len(df))
print("After license filter:", len(filtered))
print(filtered["license"].value_counts())

filtered.to_csv("candidates.csv", index=False)

Before license filter: 15739
After license filter: 9154
license
MIT License           6628
Apache License 2.0    2526
Name: count, dtype: int64


In [ ]:
from getpass import getpass
GITHUB_TOKEN = getpass("Paste the token, then press Enter: ")
print("Token captured, length:", len(GITHUB_TOKEN))

Paste the token Corey sent, then press Enter: ··········
Token captured, length: 93


In [ ]:
import base64, time, requests, pandas as pd

SAMPLE_SIZE = 300

MANIFESTS = {
    "requirements.txt","requirements-dev.txt","pyproject.toml","setup.py",
    "setup.cfg","environment.yml","environment.yaml","Pipfile","conda.yaml",
}

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

df = pd.read_csv("candidates.csv")
repos = df["name"].dropna().tolist()[:SAMPLE_SIZE]
print(f"Scanning {len(repos)} repos\n")

def root_files(full):
    r = session.get(f"https://api.github.com/repos/{full}/contents")
    return {i["name"] for i in r.json() if i.get("type") == "file"} if r.status_code == 200 else None

def mentions_mlflow(full, fname):
    r = session.get(f"https://api.github.com/repos/{full}/contents/{fname}")
    if r.status_code != 200:
        return False
    data = r.json()
    if data.get("encoding") != "base64":
        return False
    try:
        return "mlflow" in base64.b64decode(data["content"]).decode("utf-8","ignore").lower()
    except Exception:
        return False

rows = []
for i, full in enumerate(repos, 1):
    root = root_files(full)
    if root:
        for m in (MANIFESTS & root):
            if mentions_mlflow(full, m):
                rows.append({"repo": full, "evidence_file": m})
                print(f"[{i}] KEEP {full}  ({m})   total: {len(rows)}")
                break
    if i % 50 == 0:
        print(f"[{i}/{len(repos)}] ... kept so far: {len(rows)}")
    time.sleep(0.05)

result = pd.DataFrame(rows)
result.to_csv("mlflow_repos.csv", index=False)
print(f"\nDone. {len(rows)} MLflow repos out of {len(repos)} scanned")
print(f"Hit rate: {100*len(rows)/len(repos):.1f}%")

Scanning 300 repos

[4] KEEP tier4/autoware-ml  (pyproject.toml)   total: 1
[50/300] ... kept so far: 1
[100/300] ... kept so far: 1
[106] KEEP Cazzy-Aporbo/PearlMind-ML-Journey  (pyproject.toml)   total: 2
[150/300] ... kept so far: 2
[159] KEEP sunnynguyen-ai/fraud-detection-system  (requirements.txt)   total: 3
[200/300] ... kept so far: 3
[250/300] ... kept so far: 3
[300/300] ... kept so far: 3

Done. 3 MLflow repos out of 300 scanned
Hit rate: 1.0%


In [ ]:
import base64, time, os, requests, pandas as pd

MANIFESTS = {
    "requirements.txt","requirements-dev.txt","pyproject.toml","setup.py",
    "setup.cfg","environment.yml","environment.yaml","Pipfile","conda.yaml",
}

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def root_files(full):
    r = gh_get(f"https://api.github.com/repos/{full}/contents")
    return {i["name"] for i in r.json() if i.get("type") == "file"} if r.status_code == 200 else None

def mentions_mlflow(full, fname):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{fname}")
    if r.status_code != 200: return False
    d = r.json()
    if d.get("encoding") != "base64": return False
    try: return "mlflow" in base64.b64decode(d["content"]).decode("utf-8","ignore").lower()
    except Exception: return False

all_repos = pd.read_csv("candidates.csv")["name"].dropna().tolist()

processed = set()
if os.path.exists("processed.txt"):
    processed = {l.strip() for l in open("processed.txt") if l.strip()}
if not os.path.exists("mlflow_repos.csv"):
    open("mlflow_repos.csv","w").write("repo,evidence_file\n")

repos = [r for r in all_repos if r not in processed]
kept = sum(1 for _ in open("mlflow_repos.csv")) - 1
print(f"Total {len(all_repos)}, done {len(processed)}, remaining {len(repos)}, kept so far {kept}\n")

out_f = open("mlflow_repos.csv","a"); proc_f = open("processed.txt","a")
for i, full in enumerate(repos, 1):
    root = root_files(full)
    if root:
        for m in (MANIFESTS & root):
            if mentions_mlflow(full, m):
                out_f.write(f"{full},{m}\n"); out_f.flush(); kept += 1
                print(f"KEEP {full} ({m})  total kept: {kept}"); break
    proc_f.write(full + "\n"); proc_f.flush()
    if i % 200 == 0: print(f"[{i}/{len(repos)}] kept total: {kept}")
    time.sleep(0.03)
out_f.close(); proc_f.close()
print(f"\nFinished. Total MLflow repos: {kept}")

Total 9154, done 0, remaining 9154, kept so far 3

KEEP tier4/autoware-ml (pyproject.toml)  total kept: 4
KEEP Cazzy-Aporbo/PearlMind-ML-Journey (pyproject.toml)  total kept: 5
KEEP sunnynguyen-ai/fraud-detection-system (requirements.txt)  total kept: 6
[200/9154] kept total: 6
KEEP ScopeX-ASU/MAPS (requirements.txt)  total kept: 7
[400/9154] kept total: 7
KEEP hpccube/OneScience (setup.py)  total kept: 8
[600/9154] kept total: 8
KEEP Abraham-Einstein/MAFS (requirements.txt)  total kept: 9
[800/9154] kept total: 9
KEEP radixark/miles (setup.py)  total kept: 10
[1000/9154] kept total: 10
[1200/9154] kept total: 10
[1400/9154] kept total: 10
KEEP NVIDIA-AI-Blueprints/ai-model-distillation-for-financial-data (pyproject.toml)  total kept: 11
[1600/9154] kept total: 11
KEEP UpcNlp/OiaFed (pyproject.toml)  total kept: 12
KEEP hoangsonww/Spot-the-Scam-AI-Job-Fraud (pyproject.toml)  total kept: 13
[1800/9154] kept total: 13
KEEP kylejones200/geosuite (pyproject.toml)  total kept: 14
KEEP Nabid

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/mlflow_research'
os.makedirs(PROJECT, exist_ok=True)
print("Folder exists:", os.path.isdir(PROJECT))

In [ ]:
from getpass import getpass
GITHUB_TOKEN = getpass("Paste Corey's token, then press Enter: ")
print("token len:", len(GITHUB_TOKEN))

In [ ]:
import ast, base64, time, requests, csv, os
import pandas as pd

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"})

repos = pd.read_csv(f"{PROJECT}/mlflow_repos.csv")["repo"].dropna().tolist()
print(f"Running detector on {len(repos)} MLflow repos\n")

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def code_search(full):
    paths, page = [], 1
    while True:
        r = gh_get(f"https://api.github.com/search/code?q=mlflow+repo:{full}+language:python&per_page=100&page={page}")
        if r.status_code != 200:
            print(f"  search failed {full}: {r.status_code}"); break
        items = r.json().get("items", [])
        paths += [it["path"] for it in items]
        if len(items) < 100: break
        page += 1; time.sleep(7)
    return paths

def fetch(full, path):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{path}")
    if r.status_code != 200: return None
    d = r.json()
    if d.get("encoding") != "base64": return None
    try: return base64.b64decode(d["content"]).decode("utf-8","ignore")
    except Exception: return None

def analyze(src):
    has_import, n_calls = False, 0
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return ("import mlflow" in src or "from mlflow" in src), src.count("mlflow.")
    for node in ast.walk(tree):
        if isinstance(node, ast.Import) and any(n.name=="mlflow" or n.name.startswith("mlflow.") for n in node.names):
            has_import = True
        elif isinstance(node, ast.ImportFrom) and node.module and (node.module=="mlflow" or node.module.startswith("mlflow.")):
            has_import = True
        elif isinstance(node, ast.Call):
            f = node.func
            while isinstance(f, ast.Attribute):
                if isinstance(f.value, ast.Name) and f.value.id=="mlflow":
                    n_calls += 1; break
                f = f.value
    return has_import, n_calls

out = open(f"{PROJECT}/mlflow_files.csv","w", newline="")
w = csv.writer(out); w.writerow(["repo","file_path","has_import","n_calls"])
total_files = 0
for full in repos:
    print(f"\n{full}")
    paths = code_search(full)
    print(f"  {len(paths)} candidate file(s)")
    for p in paths:
        src = fetch(full, p)
        if not src: continue
        imp, calls = analyze(src)
        if imp or calls:
            w.writerow([full, p, imp, calls]); total_files += 1
            print(f"  MLflow file: {p}  (import={imp}, calls={calls})")
    time.sleep(7)
out.close()
print(f"\nDone. {total_files} MLflow files across {len(repos)} repos. Saved to Drive.")

In [ ]:
import os
print("PROJECT:", PROJECT)
print(os.listdir(PROJECT))

In [ ]:
import ast, base64, time, requests, csv, os

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"})

repos = ["tier4/autoware-ml",
         "Cazzy-Aporbo/PearlMind-ML-Journey",
         "sunnynguyen-ai/fraud-detection-system"]
print(f"Running detector on {len(repos)} MLflow repos\n")

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def code_search(full):
    paths, page = [], 1
    while True:
        r = gh_get(f"https://api.github.com/search/code?q=mlflow+repo:{full}+language:python&per_page=100&page={page}")
        if r.status_code != 200:
            print(f"  search failed {full}: {r.status_code}"); break
        items = r.json().get("items", [])
        paths += [it["path"] for it in items]
        if len(items) < 100: break
        page += 1; time.sleep(7)
    return paths

def fetch(full, path):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{path}")
    if r.status_code != 200: return None
    d = r.json()
    if d.get("encoding") != "base64": return None
    try: return base64.b64decode(d["content"]).decode("utf-8","ignore")
    except Exception: return None

def analyze(src):
    has_import, n_calls = False, 0
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return ("import mlflow" in src or "from mlflow" in src), src.count("mlflow.")
    for node in ast.walk(tree):
        if isinstance(node, ast.Import) and any(n.name=="mlflow" or n.name.startswith("mlflow.") for n in node.names):
            has_import = True
        elif isinstance(node, ast.ImportFrom) and node.module and (node.module=="mlflow" or node.module.startswith("mlflow.")):
            has_import = True
        elif isinstance(node, ast.Call):
            f = node.func
            while isinstance(f, ast.Attribute):
                if isinstance(f.value, ast.Name) and f.value.id=="mlflow":
                    n_calls += 1; break
                f = f.value
    return has_import, n_calls

out = open(f"{PROJECT}/mlflow_files.csv","w", newline="")
w = csv.writer(out); w.writerow(["repo","file_path","has_import","n_calls"])
total_files = 0
for full in repos:
    print(f"\n{full}")
    paths = code_search(full)
    print(f"  {len(paths)} candidate file(s)")
    for p in paths:
        src = fetch(full, p)
        if not src: continue
        imp, calls = analyze(src)
        if imp or calls:
            w.writerow([full, p, imp, calls]); total_files += 1
            print(f"  MLflow file: {p}  (import={imp}, calls={calls})")
    time.sleep(7)
out.close()
print(f"\nDone. {total_files} MLflow files across {len(repos)} repos. Saved to Drive.")